In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('..')

In [3]:
from sklearn.metrics import f1_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from vllm import TokensPrompt
from vllm import LLM
from pathlib import Path
from argparse import ArgumentParser

import torch
import json
import numpy as np

from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset

from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN
from cluster_intrep_repo.stacks_utils import *
from tqdm.auto import tqdm, trange

INFO 03-13 15:30:22 __init__.py:190] Automatically detected platform cuda.


In [4]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device = 'cuda'
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = initialize_tokenizer(model_id)


In [5]:
n_blocks = 6

In [6]:
parsed_datasets = {
    4: "blocksworld-4-blocks-actions-first.json",
    6: "blocksworld-6-blocks-actions-first.json"
}

In [7]:
dataset = load_dataset(
    f"dmitriihook/deepseek-r1-qwen-32b-planning-{blocksworld_type[n_blocks]}")["train"]

def load_dataset_from_file(domain_name, task_name):
    prompt_dir = Path(f"./cot-planning/results/{domain_name}/deepseek-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)


task_name = "plan_generation_po"
domain_name = f"blocksworld_{n_blocks}_blocks"
eval_results = load_dataset_from_file(domain_name, task_name)["instances"]

eval_results = {x["dataset_idx"]: x for x in eval_results}

In [8]:
with open("first_action_token_logits.json") as f:
    labels_dataset = json.load(f)

labels_dataset = {
    int(k): v for k, v in labels_dataset.items()
}

In [9]:
all_blocks = [
    chr(ord('A') + i) for i in range(n_blocks)
]

In [10]:
n_rows = row_ns[n_blocks]
n_rows = 5000

labels_dict = defaultdict(dict)

for idx, row in enumerate(dataset.select(range(n_rows))):
    generation = row["generation"]

    steps = generation.split("\n\n")[:50]

    if idx not in labels_dataset:
        continue

    for line_n, step in enumerate(steps):
        logits = []
        for block in all_blocks:
            logits.append(labels_dataset[idx]["block_logits"][block][line_n])

        logits = np.array(logits)
        smax = np.exp(logits) / np.sum(np.exp(logits))
        amax = np.argmax(logits)

        labels_dict[idx][line_n] = {
            "logits": logits,
            "probs": smax,
            "max_block": all_blocks[amax],
            "step": step
        }

In [11]:
# total_layers = model.config.num_hidden_layers
total_layers = 64

In [12]:
llm = LLM(model=model_id, task="reward", tensor_parallel_size=8)

INFO 03-13 15:30:39 config.py:1401] Defaulting to use mp for distributed inference
WARNING 03-13 15:30:39 arg_utils.py:1145] The model has a long context length (131072). This may cause OOM errors during the initial memory profiling phase, or result in low performance due to small KV cache space. Consider setting --max-model-len to a smaller value.
INFO 03-13 15:30:39 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(gu

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]


(VllmWorkerProcess pid=94855) INFO 03-13 15:31:05 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94882) INFO 03-13 15:31:05 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94870) INFO 03-13 15:31:06 model_runner.py:1115] Loading model weights took 7.5269 GB
INFO 03-13 15:31:06 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94875) INFO 03-13 15:31:06 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94865) INFO 03-13 15:31:06 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94860) INFO 03-13 15:31:07 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=94852) INFO 03-13 15:31:07 model_runner.py:1115] Loading model weights took 7.5269 GB


(VllmWorkerProcess pid=94860) (VllmWorkerProcess pid=94865) (VllmWorkerProcess pid=94870) (VllmWorkerProcess pid=94852) (VllmWorkerProcess pid=94855) INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
(VllmWorkerProcess pid=94882) INFO 03-13 15:38:15 multiproc_worker_utils.py:253] Worker exiting
(VllmWorkerProcess pid=94875) 

In [13]:
def make_data_to_process(dataset, n_rows, eval_results, answer_type, tokenizer):
    data_to_process = []
    for idx, row in enumerate(tqdm(dataset.select(range(n_rows)))):
        if eval_results[idx]["llm_correct"] and answer_type == "incorrect":
            continue
        if not eval_results[idx]["llm_correct"] and answer_type == "correct":
            continue
        generation = row["generation"]

        if idx not in labels_dict:
            continue

        # need to remove the eos token
        pos_start = len(tokenize_blocksworld_generation(tokenizer, row, "")[0][:-1])

        pos_pre = pos_start

        for line_n, line in enumerate(generation.split("\n\n")[:50]):
            line_tokens = tokenizer.tokenize("\n\n" + line + "\n\n")[1:]
            pos_post = pos_pre + len(line_tokens)

            if labels_dict[idx][line_n]["probs"].max() < 0.9:
                continue

            data_to_process.append({
                "idx": idx,
                "line_n": line_n,
                "pos_pre": pos_pre,
                "pos_post": pos_post,
                "pos_start": pos_start,
                "logits": labels_dict[idx][line_n]["logits"],
                "label": labels_dict[idx][line_n]["max_block"],
            })
            pos_pre = pos_post


    return data_to_process


In [14]:
# data_correct = make_data_to_process(dataset, n_rows, eval_results, "correct", n_blocks, labels_dict, take_prob, tokenizer)
# data_incorrect = make_data_to_process(dataset, n_rows, eval_results, "incorrect", n_blocks, labels_dict, take_prob, tokenizer)
data_all = make_data_to_process(dataset, n_rows, eval_results, "all", tokenizer)

  0%|          | 0/5000 [00:00<?, ?it/s]

In [15]:
len(data_all)

60447

In [16]:
idx_to_pos = {}

for x in data_all:
    idx_to_pos[x["idx"]] = max(idx_to_pos.get(x["idx"], 0), x["pos_post"])

In [17]:
batch_size = 200

last_hidden_states = []

for i in tqdm(range(0, n_rows, batch_size)):
    batch = dataset.select(range(i, min(i + batch_size, n_rows)))
    tokens = [tokenize_blocksworld_generation(
        tokenizer, row)[0] for row in batch]
    
    pos = [idx_to_pos.get(j, 100) for j in range(i, min(i + batch_size, n_rows))]
    tokens = [x[:p + 100] for x, p in zip(tokens, pos)]

    tokens = [TokensPrompt(prompt_token_ids=t) for t in tokens]

    output = llm.encode(tokens)

    for x in output:
        hs = x.outputs.data
        last_hidden_states.append(hs.cpu().to(torch.float16).numpy())


  0%|          | 0/25 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 200/200 [00:07<00:00, 26.88it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [18]:
collected_hidden_states = {total_layers -1 : last_hidden_states}

In [19]:
def process_data(items):
    new_items = []

    for item in items:

        block = item["label"]

        try:
            label = block2int(block, n_blocks)
        except Exception as e:
            print(e)
            continue

        new_items.append({
            "pos_pre": item["pos_pre"],
            "pos_post": item["pos_post"],
            "pos_start": item["pos_start"],
            "label": label,
            "idx":  item["idx"],
            "line_n": item["line_n"],
            "logits": item["logits"],
            "label": label
        })

    return new_items
    

In [20]:
class StepProbeDataset(Dataset):
    def __init__(self, items, n_layer, n_prev_tokens, shift_tokens):
        self.items = process_data(items)
        self.hidden_states = collected_hidden_states[n_layer]
        self.n_layer = n_layer
        self.n_prev_tokens = n_prev_tokens
        self.shift_tokens = shift_tokens

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]


        if False:
            _block_positions = block_positions[item["idx"]]
            pos_pre = item["pos_pre"]

            window_start = pos_pre - self.shift_tokens - self.n_prev_tokens
            window_end = pos_pre - self.shift_tokens + 1

            item_block_positions = []

            for block in all_blocks:
                block_block_positions = _block_positions[block]
                block_block_positions = block_block_positions[block_block_positions >= window_start]
                block_block_positions = block_block_positions[block_block_positions < window_end]

                if len(block_block_positions) > 0:
                    item_block_positions.append(block_block_positions)

            item_block_positions = np.concatenate(item_block_positions)

            return {
                "input": self.hidden_states[item["idx"]][item_block_positions],
                "labels": item["label"],
                "idx": item["idx"],
                "line_n": item["line_n"],
                "block_positions": item_block_positions
            }

        if True:
            pos_pre = item["pos_pre"]
            pos_post = item["pos_post"]
            pos_start = item["pos_start"]

            pos = pos_post

            label = item["label"]
            logits = item["logits"]

            return {
                "input": self.hidden_states[item["idx"]][pos - self.shift_tokens - self.n_prev_tokens:pos - self.shift_tokens + 1],
                # "input": self.hidden_states[item["idx"]][pos_start+ self.shift_tokens:pos_start + self.shift_tokens + self.n_prev_tokens + 1],
                # "input": self.hidden_states[item["idx"]][pos_start :pos_pre + 1],
                "labels": label,
                "logits": logits,
                "idx": item["idx"],
                "line_n": item["line_n"]
            }

            
        elif probe_type == "gru":
            post_pos = item["post_pos"]
            hidden_states = self.hidden_states[item["idx"]
                                               ][post_pos-n_prev_tokens:post_pos + 1]

            above, below = item["above"], item["below"]
            return {
                "input": hidden_states,
                "labels": state_to_label((above, below, None), self.top_block, self.bottom_block)
            }

In [69]:
class StepProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        # self.fc = torch.nn.Linear(input_size, hidden_size)
        # self.fc2 = torch.nn.Linear(hidden_size, n_blocks * (n_blocks + 2) * 2)
        # self.fc2 = torch.nn.Linear(input_size, n_blocks * (n_blocks + 2) * 2)
        self.fc2 = torch.nn.Linear(input_size, n_blocks)
        # self.dropout = torch.nn.Dropout(0.1)

    def forward(self, x):
        # x = self.fc(x)
        # x = torch.nn.functional.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x
        # return x.view(-1, n_blocks + 2, n_blocks * 2)


class GRUProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        self.gru = torch.nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x, *args):
        x, _ = self.gru(x)
        x = self.fc(x[:, -1])
        return x
    
class MultiProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_probes):
        super().__init__()
        self.probes = torch.nn.ModuleList([torch.nn.Linear(input_size, hidden_size) for _ in range(n_probes)])
        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x):
        
        for i in range(len(self.probes)):
            z_ = self.probes[i](x[:, i])
            if i == 0:
                z = z_
            else:
                z = z + z_
        z = z / len(self.probes)

        z = self.fc(z)

        return z
    
class AHProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        self.q = torch.nn.Parameter(torch.randn(hidden_size))
        self.v = torch.nn.Parameter(torch.randn(hidden_size))

        self.proj = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, 2)

    def forward(self, x, mask):
        x = self.proj(x)
        # scores = torch.einsum("bsh,h->bs", x, self.q)

        scores = x @ self.q

        # print(scores.shape, mask.shape)
        scores = scores.masked_fill(mask, -1000)
        scores = torch.nn.functional.softmax(scores, dim=-1)
        z = torch.matmul(scores.unsqueeze(1), x).squeeze(1)
        z = self.fc(z)
        return z
    
class MLHAProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_heads, n_blocks):
        super().__init__()
        self.head_dim = hidden_size // n_heads
        self.n_heads = n_heads

        self.q = torch.nn.Parameter(torch.ones(n_heads, self.head_dim))
        self.v = torch.nn.Parameter(torch.ones(n_heads, self.head_dim))

        self.proj = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, n_blocks)

    def forward(self, x, mask):
        x = self.proj(x)
        x = x.view(x.shape[0], x.shape[1], self.n_heads, self.head_dim)
        scores = torch.einsum("bshd,hd->bsh", x, self.q)

        scores = scores / np.sqrt(self.head_dim)

        scores = scores.masked_fill(mask.unsqueeze(-1), -1000)
        scores = torch.nn.functional.softmax(scores, dim=-2)
        z = torch.einsum("bshd,bsh->bhd", x, scores)
        z = z.view(z.shape[0], -1)
        z = self.fc(z)

        return z


In [109]:
training_data = data_all

train_test_split = 0.8
n_train = int(len(training_data) * train_test_split)

train_items = training_data[:n_train]
test_items = training_data[n_train:]


def collate_fn(batch):
    start_time = time.time()
    inputs = [torch.tensor(x["input"]) for x in batch]    
    masks = [torch.ones(x.shape[0], dtype=torch.bool) for x in inputs]
    # print(
    #     f"Time to create tensors: {time.time() - start_time}"
    # )
    start_time = time.time()
    # inputs = pad_sequence(inputs, batch_first=True,
    #                       padding_value=0, padding_side="left")
    # masks = pad_sequence(masks, batch_first=True,
    #                      padding_value=True, padding_side="left")

    inputs = torch.stack(inputs)
    masks = torch.stack(masks)
    
    # print(
    #     f"Time to pad tensors: {time.time() - start_time}"
    # )

    start_time = time.time()

    labels = np.stack([x["labels"] for x in batch])
    labels = torch.tensor(labels, dtype=torch.int64)

    logits = np.stack([x["logits"] for x in batch])
    logits = torch.tensor(logits, dtype=torch.float32)

    # print(
    #     f"Time to stack labels: {time.time() - start_time}"
    # )
    return {
        "input": inputs,
        "labels": labels,
        "logits": logits,
        "mask": masks
    }

def optimized_collate_fn(batch):
    batch_size = len(batch)
    # Assuming all inputs have the same shape
    input_shape = batch[0]["input"].shape
    
    # Pre-allocate arrays
    inputs = np.zeros((batch_size, *input_shape), dtype=np.float32)
    labels = np.zeros(batch_size, dtype=np.int64)
    logits = np.zeros((batch_size, len(batch[0]["logits"])), dtype=np.float32)
    
    # Fill arrays
    for i, item in enumerate(batch):
        inputs[i] = item["input"]
        labels[i] = item["labels"]
        logits[i] = item["logits"]
    
    # Convert to tensors once at the end
    return {
        "input": torch.from_numpy(inputs),
        "labels": torch.from_numpy(labels),
        "logits": torch.from_numpy(logits),
        "mask": torch.zeros((batch_size, input_shape[0]), dtype=torch.bool)  # No padding needed
    }

import time
batch_times = []
forward_times = []

def train_probe(probe, train_dataset, test_dataset, patience=30, lr=1e-4, n_epochs=500, batch_size=128):
    optimizer = Adam(probe.parameters(), lr=lr)
    criterion = CrossEntropyLoss()
    # criterion = torch.nn.MSELoss()
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, drop_last=False, num_workers=10, persistent_workers=True)
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, drop_last=False, num_workers=10, persistent_workers=True)

    best_f1 = float('-inf')
    early_stop_counter = 0

    for epoch in range(n_epochs):
        probe.train()
        total_loss = 0
        n_samples = 0
        epoch_start = time.time()

        prev_time = time.time()
        for batch in train_loader:
            next_time = time.time()

            batch_times.append(next_time - prev_time)

            # print(f"Mean batch time: {np.mean(batch_times[-5:])}")


            optimizer.zero_grad()
            input = batch["input"].to(device).float()
            labels = batch["labels"].to(device)
            # labels = batch["logits"].to(device)
            mask = batch["mask"].to(device)

            fwd_start = time.time()

            output = probe(input, mask)

            fwd_end = time.time()

            forward_times.append(fwd_end - fwd_start)

            # print(output.shape, labels.shape, input.shape)

            loss = criterion(output, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch["input"])
            n_samples += len(batch["input"])

            # print(f"Mean forward time: {np.mean(forward_times[-5:])}")

            prev_time = next_time

        epoch_end = time.time()
        print(f"Epoch time: {epoch_end - epoch_start}")

        avg_train_loss = total_loss / n_samples

        # Evaluation
        probe.eval()
        with torch.no_grad():
            # block_wise_hits = np.zeros((n_blocks * 2), dtype=np.int64)
            block_wise_hits = 0
            total = 0
            val_loss = 0
            all_preds = []
            all_labels = []

            for batch in test_loader:
                # print(batch["input"].shape)
                input = batch["input"].to(device).float()
                labels = batch["labels"].to(device)
                # labels = batch["logits"].to(device)
                mask = batch["mask"].to(device)

                output = probe(input, mask)

                loss = criterion(output, labels)
                val_loss += loss.item() * len(batch["input"])

                labels = batch["labels"].to(device)

                preds = output.argmax(dim=1)  # Assuming classification task
                hits = (preds == labels)

                block_wise_hits += hits.sum(dim=0).cpu().numpy()
                total += len(labels)

                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

            block_wise_hits = block_wise_hits / total

            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)

            # Compute F1 score block-wise
            # block_wise_f1 = np.zeros(n_blocks * 2)
            # for i in range(n_blocks * 2):
            #     block_wise_f1[i] = f1_score(all_labels[:, i], all_preds[:, i], average='macro')

            # avg_f1 = block_wise_f1.mean()
            avg_f1 = f1_score(all_labels, all_preds, average='macro')

            val_loss /= total

            print(
                f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Hits: {block_wise_hits.mean():.4f}, F1: {avg_f1:.4f}, Val Loss: {val_loss:.4f}")

            # Early Stopping Check
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    return block_wise_hits, best_f1

In [116]:
from torch.nn.parallel import DataParallel

n_layer = total_layers - 1

n_prev_tokens = 100
n_shift_tokens= 0

train_dataset = StepProbeDataset(
   train_items, n_layer, n_prev_tokens, n_shift_tokens)
test_dataset = StepProbeDataset(test_items, n_layer, n_prev_tokens, n_shift_tokens)

print(len(train_dataset))

n_dim = 5120
# probe = GRUProbe(n_dim, 1000, n_blocks).to(device)
probe = MLHAProbe(n_dim, n_dim, 40, n_blocks).to(device)

48357


In [117]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [118]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    hits, f1 = train_probe(probe, train_dataset, test_dataset, patience=30, lr=1e-3, n_epochs=10, batch_size=128)

Epoch time: 33.095271587371826
Epoch 0, Train Loss: 2.7509, Hits: 0.2873, F1: 0.1518, Val Loss: 1.7105
Epoch time: 26.694265365600586
Epoch 1, Train Loss: 1.6759, Hits: 0.3228, F1: 0.1591, Val Loss: 1.7690
Epoch time: 27.438583374023438
Epoch 2, Train Loss: 1.6446, Hits: 0.2141, F1: 0.1658, Val Loss: 1.8191
Epoch time: 27.43216347694397
Epoch 3, Train Loss: 1.6258, Hits: 0.3306, F1: 0.1892, Val Loss: 1.8025
Epoch time: 27.5962131023407
Epoch 4, Train Loss: 1.6472, Hits: 0.3038, F1: 0.1304, Val Loss: 1.9726


KeyboardInterrupt: 

In [119]:
stats = pstats.Stats(pr)
stats.sort_stats(pstats.SortKey.TIME)
stats.print_stats(10) 

         3659716 function calls (3615785 primitive calls) in 185.134 seconds

   Ordered by: internal time
   List reduced from 686 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    23286   62.316    0.003   62.316    0.003 {method 'item' of 'torch._C.TensorBase' objects}
     8120   58.646    0.007   58.646    0.007 {method 'to' of 'torch._C.TensorBase' objects}
        1   41.717   41.717  185.134  185.134 /tmp/ipykernel_92131/2838571172.py:76(train_probe)
       20   10.939    0.547   10.939    0.547 {built-in method posix.fork}
     2639    2.045    0.001    2.045    0.001 {method 'poll' of 'select.poll' objects}
     2073    2.013    0.001    2.013    0.001 {method 'run_backward' of 'torch._C._EngineBase' objects}
    66454    1.216    0.000    1.216    0.000 {built-in method posix.read}
     5096    0.367    0.000    0.367    0.000 {built-in method torch._C._nn.linear}
     5096    0.349    0.000    0.349    0.000 {built-in

In [68]:
stats = pstats.Stats(pr)
stats.sort_stats(pstats.SortKey.TIME)
stats.print_stats(10) 

         3697654 function calls (3694620 primitive calls) in 695.164 seconds

   Ordered by: internal time
   List reduced from 491 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      300  309.398    1.031  309.398    1.031 {built-in method torch.stack}
   604895  178.210    0.000  178.245    0.000 {built-in method torch.tensor}
     1380  115.359    0.084  115.359    0.084 {method 'item' of 'torch._C.TensorBase' objects}
      480   70.346    0.147   70.346    0.147 {method 'to' of 'torch._C.TensorBase' objects}
        1   14.528   14.528  695.165  695.165 /tmp/ipykernel_92131/3744117582.py:53(train_probe)
   604470    1.452    0.000    1.471    0.000 /tmp/ipykernel_92131/1033219346.py:12(__getitem__)
      150    1.081    0.007    1.914    0.013 /tmp/ipykernel_92131/3744117582.py:13(<listcomp>)
      150    0.910    0.006  179.080    1.194 /tmp/ipykernel_92131/3744117582.py:12(<listcomp>)
   604470    0.818    0.000    0.818  

In [59]:

block_wise_hits, best_f1 = train_probe(
    probe, train_dataset, test_dataset, patience=10, lr=3e-5)

print(best_f1)

Time to create tensors: 1.5131754875183105
Time to pad tensors: 2.140195846557617
Time to stack labels: 0.0077092647552490234
Mean batch time: 4.51773476600647


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.89 GiB. GPU 0 has a total capacity of 79.10 GiB of which 7.46 GiB is free. Including non-PyTorch memory, this process has 71.61 GiB memory in use. Of the allocated memory 56.65 GiB is allocated by PyTorch, and 9.47 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [66]:
torch.cuda.empty_cache()

import gc

gc.collect()

200